# NER Ensemble — NB1b + NB1b_pubmed

Standalone notebook — no training, no inference, no GPU needed.

Loads two existing prediction files, tests multiple ensemble strategies,
evaluates with the **exact** `evaluate.py` logic, and saves the best result.

| File | Model |
|------|-------|
| `pred_NB1b_gold_silver_bronze.json` | BioBERT + Gold + Silver + Bronze |
| `pred_NB1b_pubmed_gold_silver_bronze.json` | PubMedBERT + Gold + Silver + Bronze |

## 0. Imports & paths

In [1]:
import json
import copy
import pandas as pd
from pathlib import Path
from collections import Counter

# ── Paths (same layout as inference_all_models.ipynb) ─────────────────────────
def find_repo_root(start: Path) -> Path:
    for p in [start] + list(start.parents):
        if (p / 'data').exists() and (p / 'src').exists():
            return p
    raise FileNotFoundError('Cannot find repo root')

PROJECT_ROOT = find_repo_root(Path.cwd())
PRED_DIR = PROJECT_ROOT / 'src' / 'ner' / 'predictions'
DEV_PATH = (
    PROJECT_ROOT / 'data' / 'GutBrainIE_Full_Collection_2026'
    / 'Annotations' / 'Dev' / 'json_format' / 'dev.json'
)

# ── Prediction files to ensemble ──────────────────────────────────────────────
PRED_A_PATH = PRED_DIR / 'pred_NB1b_gold_silver_bronze.json'         # BioBERT
PRED_B_PATH = PRED_DIR / 'pred_NB1b_pubmed_gold_silver_bronze.json'  # PubMedBERT

print('Project root:', PROJECT_ROOT)
print('Dev file exists  :', DEV_PATH.exists())
print('Pred A (NB1b)    :', PRED_A_PATH.exists(), '—', PRED_A_PATH.name)
print('Pred B (pubmed)  :', PRED_B_PATH.exists(), '—', PRED_B_PATH.name)

Project root: C:\Users\super\Documents\UniPd\ATA\SMTE-GutBrainIE
Dev file exists  : True
Pred A (NB1b)    : True — pred_NB1b_gold_silver_bronze.json
Pred B (pubmed)  : True — pred_NB1b_pubmed_gold_silver_bronze.json


## 1. Load predictions & dev ground truth

In [2]:
with PRED_A_PATH.open(encoding='utf-8') as f:
    pred_biobert = json.load(f)
with PRED_B_PATH.open(encoding='utf-8') as f:
    pred_pubmed = json.load(f)
with DEV_PATH.open(encoding='utf-8') as f:
    dev_data = json.load(f)

print(f'BioBERT  — PMIDs: {len(pred_biobert):3d}  entities: {sum(len(v["entities"]) for v in pred_biobert.values())}')
print(f'PubMedBERT— PMIDs: {len(pred_pubmed):3d}  entities: {sum(len(v["entities"]) for v in pred_pubmed.values())}')
print(f'Dev gold — PMIDs: {len(dev_data):3d}')

BioBERT  — PMIDs:  80  entities: 2356
PubMedBERT— PMIDs:  80  entities: 2287
Dev gold — PMIDs:  80


## 2. Official evaluator — exact copy of `evaluate.py`

Two critical details:
- dedup key = `(start_idx, end_idx, location)` — **no label**
- overlap condition = `start_idx < current_end` — **strict** less-than

In [3]:
LEGAL_ENTITY_LABELS = [
    'anatomical location', 'animal', 'bacteria', 'biomedical technique',
    'chemical', 'DDF', 'dietary supplement', 'drug', 'food', 'gene',
    'human', 'microbiome', 'statistical technique'
]


def remove_duplicated_entities(predictions: dict) -> None:
    """In-place. Key = (start_idx, end_idx, location) — no label."""
    removed = 0
    for pmid in list(predictions):
        seen, deduped = set(), []
        for e in predictions[pmid]['entities']:
            k = (e['start_idx'], e['end_idx'], e['location'])
            if k not in seen: seen.add(k); deduped.append(e)
            else: removed += 1
        predictions[pmid]['entities'] = deduped
    if removed > 0:
        print(f'=== Removed {removed} duplicated entities ===')


def remove_overlapping_entities(predictions: dict) -> None:
    """In-place. Overlap = start_idx < current_end (strict)."""
    removed = 0
    for pmid in list(predictions):
        orig = len(predictions[pmid]['entities'])
        groups = {'title': [], 'abstract': []}
        for e in predictions[pmid]['entities']:
            groups[e['location']].append(e)
        keepers = set()
        for loc in groups:
            group = sorted(groups[loc], key=lambda e: e['start_idx'])
            clusters, cluster, cur_end = [], [], None
            for e in group:
                if not cluster:
                    cluster = [e]; cur_end = e['end_idx']
                elif e['start_idx'] < cur_end:
                    cluster.append(e)
                    if e['end_idx'] > cur_end: cur_end = e['end_idx']
                else:
                    clusters.append(cluster); cluster = [e]; cur_end = e['end_idx']
            if cluster: clusters.append(cluster)
            for cl in clusters:
                longest = cl[0]
                for e in cl[1:]:
                    if e['end_idx'] - e['start_idx'] > longest['end_idx'] - longest['start_idx']:
                        longest = e
                keepers.add((longest['start_idx'], longest['end_idx'], longest['location']))
        deduped = []
        for e in predictions[pmid]['entities']:
            k = (e['start_idx'], e['end_idx'], e['location'])
            if k in keepers: deduped.append(e); keepers.discard(k)
        predictions[pmid]['entities'] = deduped
        removed += orig - len(deduped)
    if removed > 0:
        print(f'=== Removed {removed} overlapping entities ===')


def evaluate_official(predictions: dict, ground_truth: dict) -> dict:
    """Exact replica of eval_submission_NER() from evaluate.py."""
    preds = copy.deepcopy(predictions)  # never modify the original
    remove_duplicated_entities(preds)
    remove_overlapping_entities(preds)

    ground_truth_NER = {}
    count_gold = {}
    for pmid, article in ground_truth.items():
        ground_truth_NER[pmid] = []
        for e in article['entities']:
            lab = str(e['label'])
            entry = (int(e['start_idx']), int(e['end_idx']),
                     str(e['location']), str(e['text_span']), lab)
            ground_truth_NER[pmid].append(entry)
            count_gold[lab] = count_gold.get(lab, 0) + 1

    count_pred = {l: 0 for l in count_gold}
    count_tp   = {l: 0 for l in count_gold}
    for pmid, obj in preds.items():
        for e in obj.get('entities', []):
            lab = str(e['label'])
            if lab not in LEGAL_ENTITY_LABELS:
                raise NameError(f'{pmid} - Illegal label: {lab}')
            count_pred[lab] = count_pred.get(lab, 0) + 1
            entry = (int(e['start_idx']), int(e['end_idx']),
                     str(e['location']), str(e['text_span']), lab)
            if entry in ground_truth_NER.get(pmid, []):
                count_tp[lab] += 1

    total_gold = sum(count_gold.values())
    total_pred = sum(count_pred.values())
    total_tp   = sum(count_tp.values())
    micro_p  = total_tp / (total_pred + 1e-10)
    micro_r  = total_tp / (total_gold + 1e-10)
    micro_f1 = 2 * micro_p * micro_r / (micro_p + micro_r + 1e-10)

    macro_p = macro_r = macro_f1 = 0.0
    per_label = {}
    n = 0
    for lab in count_gold:
        n += 1
        p  = count_tp[lab] / (count_pred[lab] + 1e-10)
        r  = count_tp[lab] / (count_gold[lab] + 1e-10)
        f1 = 2 * p * r / (p + r + 1e-10)
        macro_p += p; macro_r += r; macro_f1 += f1
        per_label[lab] = {
            'P': round(p, 4), 'R': round(r, 4), 'F1': round(f1, 4),
            'TP': count_tp[lab], 'pred': count_pred[lab], 'gold': count_gold[lab]
        }

    return {
        'macro_P':  round(macro_p  / n, 4),
        'macro_R':  round(macro_r  / n, 4),
        'macro_F1': round(macro_f1 / n, 4),
        'micro_P':  round(micro_p,      4),
        'micro_R':  round(micro_r,      4),
        'micro_F1': round(micro_f1,     4),
        'per_label': per_label,
    }


print('Official evaluator ready.')

Official evaluator ready.


## 3. Ensemble strategies

| Strategy | Logic |
|----------|-------|
| **Union** | all spans from both models |
| **Intersection** | only spans both agree on (span + label) |
| **Hybrid A** | PubMedBERT base + NB1b extras on recall-weak labels |
| **Hybrid B** | PubMedBERT base + NB1b extras on ALL labels |
| **Hybrid C** | Union but prefer PubMedBERT label on span conflicts |

In [4]:
RECALL_LABELS = {'food', 'gene', 'statistical technique', 'bacteria', 'chemical'}


def build_union(a, b):
    """Keep every span predicted by either model."""
    result = {}
    for pmid in a:
        seen, ents = set(), []
        for e in a[pmid]['entities'] + b[pmid]['entities']:
            k = (e['start_idx'], e['end_idx'], e['location'], e['label'])
            if k not in seen: seen.add(k); ents.append(e)
        result[pmid] = {'entities': ents}
    return result


def build_intersection(a, b):
    """Keep only spans where both models agree (same span + same label)."""
    result = {}
    for pmid in a:
        set_b = set(
            (e['start_idx'], e['end_idx'], e['location'], e['label'])
            for e in b[pmid]['entities']
        )
        ents = [
            e for e in a[pmid]['entities']
            if (e['start_idx'], e['end_idx'], e['location'], e['label']) in set_b
        ]
        result[pmid] = {'entities': ents}
    return result


def build_hybrid_recall(base, extra, recall_labels):
    """Base model + extra spans from 'extra' only for recall_labels."""
    result = {}
    for pmid in base:
        covered = set(
            (e['start_idx'], e['end_idx'], e['location'])
            for e in base[pmid]['entities']
        )
        new = [
            e for e in extra[pmid]['entities']
            if e['label'] in recall_labels
            and (e['start_idx'], e['end_idx'], e['location']) not in covered
        ]
        result[pmid] = {'entities': base[pmid]['entities'] + new}
    return result


def build_hybrid_all(base, extra):
    """Base model + ALL extra spans not already covered by base."""
    result = {}
    for pmid in base:
        covered = set(
            (e['start_idx'], e['end_idx'], e['location'])
            for e in base[pmid]['entities']
        )
        new = [
            e for e in extra[pmid]['entities']
            if (e['start_idx'], e['end_idx'], e['location']) not in covered
        ]
        result[pmid] = {'entities': base[pmid]['entities'] + new}
    return result


def build_hybrid_label_vote(pubmed, biobert):
    """
    Union of spans, but on label conflicts prefer PubMedBERT.
    If only one model predicts a span, keep that model's label.
    """
    result = {}
    for pmid in pubmed:
        # pubmed spans: span -> entity
        pub_map = {
            (e['start_idx'], e['end_idx'], e['location']): e
            for e in pubmed[pmid]['entities']
        }
        bio_map = {
            (e['start_idx'], e['end_idx'], e['location']): e
            for e in biobert[pmid]['entities']
        }
        all_spans = set(pub_map) | set(bio_map)
        ents = []
        for span in all_spans:
            if span in pub_map:
                ents.append(pub_map[span])   # pubmed wins on conflicts
            else:
                ents.append(bio_map[span])   # biobert fills what pubmed missed
        result[pmid] = {'entities': ents}
    return result


print('Ensemble functions defined.')

Ensemble functions defined.


## 4. Evaluate all strategies

In [5]:
strategies = {
    'A — BioBERT only':              pred_biobert,
    'B — PubMedBERT only':           pred_pubmed,
    'C — Union':                     build_union(pred_biobert, pred_pubmed),
    'D — Intersection':              build_intersection(pred_biobert, pred_pubmed),
    'E — Hybrid recall labels':      build_hybrid_recall(pred_pubmed, pred_biobert, RECALL_LABELS),
    'F — Hybrid all labels':         build_hybrid_all(pred_pubmed, pred_biobert),
    'G — Label vote (pubmed wins)':  build_hybrid_label_vote(pred_pubmed, pred_biobert),
}

results = {}
rows = []

print(f"{'Strategy':<35} {'Macro-P':>8} {'Macro-R':>8} {'Macro-F1':>9} {'Micro-F1':>9}")
print('-' * 75)

for name, preds in strategies.items():
    m = evaluate_official(preds, dev_data)
    results[name] = m
    rows.append({
        'Strategy': name,
        'Macro-P':  m['macro_P'],
        'Macro-R':  m['macro_R'],
        'Macro-F1': m['macro_F1'],
        'Micro-F1': m['micro_F1'],
    })
    print(f"{name:<35} {m['macro_P']:>8.4f} {m['macro_R']:>8.4f} "
          f"{m['macro_F1']:>9.4f} {m['micro_F1']:>9.4f}")

df = pd.DataFrame(rows).set_index('Strategy')
best_macro = df['Macro-F1'].idxmax()
best_micro = df['Micro-F1'].idxmax()
print(f'\nBest Macro-F1: {best_macro} ({df["Macro-F1"].max():.4f})')
print(f'Best Micro-F1: {best_micro} ({df["Micro-F1"].max():.4f})')

Strategy                             Macro-P  Macro-R  Macro-F1  Micro-F1
---------------------------------------------------------------------------
A — BioBERT only                      0.8091   0.7587    0.7749    0.8194
B — PubMedBERT only                   0.8305   0.7728    0.7973    0.8270
=== Removed 7 duplicated entities ===
=== Removed 43 overlapping entities ===
C — Union                             0.8005   0.8049    0.7986    0.8312
D — Intersection                      0.8632   0.7204    0.7772    0.8190
=== Removed 19 overlapping entities ===
E — Hybrid recall labels              0.8218   0.7865    0.7998    0.8276
=== Removed 43 overlapping entities ===
F — Hybrid all labels                 0.8040   0.8068    0.8020    0.8324
=== Removed 43 overlapping entities ===
G — Label vote (pubmed wins)          0.8040   0.8068    0.8020    0.8324

Best Macro-F1: F — Hybrid all labels (0.8020)
Best Micro-F1: F — Hybrid all labels (0.8324)


## 5. Per-label F1 comparison

In [6]:
label_rows = []
for name, m in results.items():
    for lab, vals in m['per_label'].items():
        label_rows.append({'Strategy': name, 'Label': lab, 'F1': vals['F1']})

df_labels = pd.DataFrame(label_rows)
df_pivot  = df_labels.pivot(index='Label', columns='Strategy', values='F1').round(4)
df_pivot['Best'] = df_pivot.idxmax(axis=1)

display(df_pivot.style
    .highlight_max(
        subset=[c for c in df_pivot.columns if c != 'Best'],
        color='#d4edda', axis=1)
    .format('{:.4f}', subset=[c for c in df_pivot.columns if c != 'Best'])
    .set_caption('Per-label F1 — ensemble comparison'))

Strategy,A — BioBERT only,B — PubMedBERT only,C — Union,D — Intersection,E — Hybrid recall labels,F — Hybrid all labels,G — Label vote (pubmed wins),Best
Label,,,,,,,,
DDF,0.8928,0.8775,0.8962,0.8750,0.8775,0.8962,0.8962,C — Union
anatomical location,0.7584,0.8324,0.7947,0.7914,0.8324,0.7947,0.7947,B — PubMedBERT only
animal,0.8515,0.8232,0.8562,0.8443,0.8232,0.8562,0.8562,C — Union
bacteria,0.7478,0.7418,0.7670,0.7323,0.7670,0.7670,0.7670,C — Union
biomedical technique,0.6980,0.6996,0.6980,0.7025,0.6996,0.6980,0.6980,D — Intersection
chemical,0.7221,0.7383,0.7355,0.7287,0.7370,0.7370,0.7370,B — PubMedBERT only
dietary supplement,0.7667,0.7857,0.7805,0.7706,0.7857,0.7934,0.7934,F — Hybrid all labels
drug,0.8553,0.8497,0.8571,0.8591,0.8497,0.8625,0.8625,F — Hybrid all labels
food,0.6136,0.7455,0.7619,0.5882,0.7818,0.7963,0.7963,F — Hybrid all labels


## 6. Save best ensemble

In [7]:
# ── Pick the best strategy by Macro-F1 ────────────────────────────────────────
best_name = df['Macro-F1'].idxmax()
best_preds = strategies[best_name]
best_metrics = results[best_name]

print(f'Best strategy : {best_name}')
print(f'Macro-F1      : {best_metrics["macro_F1"]}')
print(f'Micro-F1      : {best_metrics["micro_F1"]}')

# ── Save ──────────────────────────────────────────────────────────────────────
out_path = PRED_DIR / 'pred_ensemble_best.json'
with out_path.open('w', encoding='utf-8') as f:
    json.dump(best_preds, f, ensure_ascii=False, indent=2)
print(f'\nSaved → {out_path}')

# ── Also save per-strategy for reference ──────────────────────────────────────
strategy_to_filename = {
    'C — Union':                    'pred_ensemble_union.json',
    'D — Intersection':             'pred_ensemble_intersection.json',
    'E — Hybrid recall labels':     'pred_ensemble_hybrid_recall.json',
    'F — Hybrid all labels':        'pred_ensemble_hybrid_all.json',
    'G — Label vote (pubmed wins)': 'pred_ensemble_label_vote.json',
}
for name, fname in strategy_to_filename.items():
    p = PRED_DIR / fname
    with p.open('w', encoding='utf-8') as f:
        json.dump(strategies[name], f, ensure_ascii=False, indent=2)
    print(f'Saved {fname}')

Best strategy : F — Hybrid all labels
Macro-F1      : 0.802
Micro-F1      : 0.8324

Saved → C:\Users\super\Documents\UniPd\ATA\SMTE-GutBrainIE\src\ner\predictions\pred_ensemble_best.json
Saved pred_ensemble_union.json
Saved pred_ensemble_intersection.json
Saved pred_ensemble_hybrid_recall.json
Saved pred_ensemble_hybrid_all.json
Saved pred_ensemble_label_vote.json
